# xLSTM + PPO: сравнение трёх архитектур

| Модель | Суть |
|--------|------|
| LSTM baseline | обычный LSTM(128) |
| xLSTM Base | xLSTM(emb=128, blocks=4) |
| xLSTM+Attention | xLSTM(128,4) + MultiheadAttention |
| xLSTM Large | xLSTM(emb=256, blocks=6) |

Все улучшения: нормализация по train, 9 индикаторов, улучшенный reward, curriculum learning, margin=0.25

In [ ]:
!pip install stable-baselines3 sb3-contrib pandas numpy requests apimoex matplotlib pandas_ta -q
!pip install git+https://github.com/NX-AI/xlstm.git -q

In [ ]:
import pandas as pd
import numpy as np
import requests
import apimoex
import pandas_ta as ta
import warnings
warnings.filterwarnings('ignore')

## Шаг 1: Загрузка данных

In [ ]:
TICKERS=['SBER','GAZP','LKOH','YNDX','GMKN']
TRAIN_START='2009-01-01'; TRAIN_END='2021-12-31'
TEST_START='2022-01-01'; TEST_END='2024-01-01'
DATA_FILE='moex_data.csv'

def download_moex_ohlcv(ticker, start, end):
    with requests.Session() as s:
        data = apimoex.get_board_history(s, ticker, start=start, end=end,
            columns=('TRADEDATE','OPEN','HIGH','LOW','CLOSE','VOLUME'))
    if not data: raise ValueError(f'Нет данных для {ticker}')
    df = pd.DataFrame(data)
    df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
    df = df.set_index('TRADEDATE')
    df.columns = [c.lower() for c in df.columns]
    df = df.add_prefix(f'{ticker}_')
    return df.replace(0, np.nan)

def load_or_download(tickers, start, end, cache):
    import os
    if os.path.exists(cache):
        print(f'Кэш: {cache}')
        return pd.read_csv(cache, index_col=0, parse_dates=True)
    frames=[]
    for t in tickers:
        print(f'  {t}...', end=' ', flush=True)
        frames.append(download_moex_ohlcv(t, start, end))
        print('OK')
    df=pd.concat(frames,axis=1).ffill().bfill()
    df.to_csv(cache); return df

raw = load_or_download(TICKERS, TRAIN_START, TEST_END, DATA_FILE)
print(f'Загружено: {raw.shape[0]} дней, {raw.shape[1]} колонок')

## Шаг 2: Features + Turbulence (нормализация только по train)

In [ ]:
def extract_close_prices(df, tickers):
    return df[[f'{t}_close' for t in tickers]].copy()

def compute_turbulence(close_prices, lookback=252):
    returns = close_prices.pct_change().dropna()
    turb = pd.Series(index=returns.index, dtype=float, name='turbulence')
    for i in range(lookback, len(returns)):
        hist = returns.iloc[i-lookback:i]
        y = returns.iloc[i].values - hist.mean().values
        try: t = float(y @ np.linalg.pinv(hist.cov().values) @ y)
        except: t = 0.0
        turb.iloc[i] = max(t, 0.0)
    return turb.fillna(0.0)

def compute_features(df, tickers, fit_mask=None):
    parts = []
    for t in tickers:
        close=df[f'{t}_close']; high=df[f'{t}_high']
        low=df[f'{t}_low']; volume=df[f'{t}_volume']
        sub = df[[f'{t}_{c}' for c in ['open','high','low','close','volume']]].copy()
        sub[f'{t}_trend']  = (ta.ema(close,length=10)-ta.ema(close,length=30))/(close.abs()+1e-8)
        sub[f'{t}_adx']    = ta.adx(high,low,close,length=14)['ADX_14']/100
        sub[f'{t}_rsi']    = ta.rsi(close,length=14)/100
        macd=ta.macd(close); sub[f'{t}_macd'] = macd['MACD_12_26_9']/(close.abs()+1e-8)
        sub[f'{t}_mom']    = ta.mom(close,length=10)/(close.abs()+1e-8)
        bb=ta.bbands(close,length=20)
        sub[f'{t}_bb_pct'] = (close-bb['BBL_20_2.0'])/(bb['BBU_20_2.0']-bb['BBL_20_2.0']+1e-8)
        sub[f'{t}_atr']    = ta.atr(high,low,close,length=14)/(close.abs()+1e-8)
        sub[f'{t}_obv']    = ta.obv(close,volume)
        sub[f'{t}_mfi']    = ta.mfi(high,low,close,volume,length=14)/100
        sub = sub.fillna(0)
        if fit_mask is not None:
            mean=sub[fit_mask].mean(); std=sub[fit_mask].std()+1e-8
        else:
            mean=sub.mean(); std=sub.std()+1e-8
        sub=(sub-mean)/std; parts.append(sub.values)
    return np.concatenate(parts,axis=1).astype(np.float32)

close_prices = extract_close_prices(raw, TICKERS)
print('Turbulence (~1-2 мин)...')
turbulence   = compute_turbulence(close_prices)
TURBULENCE_THRESHOLD = float(np.percentile(turbulence[TRAIN_START:TRAIN_END].dropna(),75))
print(f'Threshold: {TURBULENCE_THRESHOLD:.2f}')

common_idx   = raw.index.intersection(turbulence.dropna().index)
raw_aligned  = raw.loc[common_idx]; turb_aligned=turbulence.loc[common_idx]
dates_all    = raw_aligned.index
train_mask   = (dates_all>=TRAIN_START)&(dates_all<=TRAIN_END)
test_mask    = (dates_all>=TEST_START) &(dates_all<=TEST_END)
calm_mask    = (dates_all>='2013-01-01')&(dates_all<='2019-12-31')

print('Индикаторы...')
features_all   = compute_features(raw_aligned, TICKERS, fit_mask=train_mask)
close_all      = close_prices.loc[common_idx].values.astype(np.float32)
features_train = features_all[train_mask]; close_train=close_all[train_mask]
turb_train_arr = turb_aligned[train_mask].values
features_test  = features_all[test_mask];  close_test=close_all[test_mask]
turb_test_arr  = turb_aligned[test_mask].values
print(f'Train: {features_train.shape[0]} | Test: {features_test.shape[0]} | Features: {features_train.shape[1]}')

## Шаг 3: Торговые среды

In [ ]:
import gymnasium as gym
from gymnasium import spaces

class MOEXTradingEnv(gym.Env):
    metadata={'render_modes':[]}
    def __init__(self,close_prices,features,turbulence,turbulence_threshold,
                 initial_balance=1_000_000.,window_size=30,commission=0.0005,max_shares_per_trade=100):
        super().__init__()
        self.close_prices=close_prices; self.features=features
        self.turbulence=turbulence; self.turbulence_threshold=turbulence_threshold
        self.initial_balance=initial_balance; self.window_size=window_size
        self.commission=commission; self.max_shares=max_shares_per_trade
        self.n_assets=close_prices.shape[1]; self.n_features=features.shape[1]
        self.action_space=spaces.Box(low=-1.,high=1.,shape=(self.n_assets,),dtype=np.float32)
        self.observation_space=spaces.Box(low=-np.inf,high=np.inf,shape=(window_size,self.n_features),dtype=np.float32)
    def reset(self,seed=None,options=None):
        super().reset(seed=seed)
        self.current_step=self.window_size; self.balance=self.initial_balance
        self.shares_held=np.zeros(self.n_assets,dtype=np.float32)
        self.prev_portfolio_value=self.initial_balance
        self.portfolio_history=[self.initial_balance]; self.trade_count=0
        return self._get_obs(),{}
    def step(self,action):
        turb=self.turbulence[self.current_step]
        if turb>self.turbulence_threshold:
            self.current_step+=1; done=self.current_step>=len(self.close_prices)
            self.portfolio_history.append(self._portfolio_value())
            return self._get_obs(),-1.,done,False,{'turbulence':turb}
        prices=(self.close_prices[self.current_step])
        sd=(action*self.max_shares).astype(np.float32)
        sd=np.maximum(sd,-self.shares_held)
        buy=np.sum(np.maximum(sd,0)*prices)
        if buy>self.balance: sd=np.where(sd>0,sd*self.balance/(buy+1e-8),sd)
        cost=np.sum(np.abs(sd)*prices)*self.commission
        self.balance-=np.sum(sd*prices)+cost; self.shares_held+=sd
        self.trade_count+=int(np.any(sd!=0)); self.current_step+=1
        done=self.current_step>=len(self.close_prices)
        nv=self._portfolio_value(); self.portfolio_history.append(nv)
        r=float(np.clip((nv-self.prev_portfolio_value)/(self.prev_portfolio_value+1e-8),-1.,1.))
        self.prev_portfolio_value=nv
        return self._get_obs(),r,done,False,{'portfolio_value':nv,'turbulence':turb}
    def render(self):
        v=self._portfolio_value()
        print(f'День {self.current_step:4d} | {v:>14,.0f} ₽ | {(v/self.initial_balance-1)*100:+.2f}%')
    def _get_obs(self): return self.features[self.current_step-self.window_size:self.current_step].copy()
    def _portfolio_value(self):
        idx=min(self.current_step,len(self.close_prices)-1)
        return float(self.balance+np.sum(self.shares_held*self.close_prices[idx]))

class MOEXTradingEnvShort(MOEXTradingEnv):
    """Short selling + улучшенный reward + автозакрытие шорта при росте >2%."""
    def __init__(self,*args,margin_ratio=0.25,**kwargs):
        super().__init__(*args,**kwargs); self.margin_ratio=margin_ratio; self._prev_prices=None
    def reset(self,seed=None,options=None):
        obs,info=super().reset(seed=seed,options=options); self._prev_prices=None; return obs,info
    def step(self,action):
        turb=self.turbulence[self.current_step]
        if turb>self.turbulence_threshold:
            self.current_step+=1; done=self.current_step>=len(self.close_prices)
            self.portfolio_history.append(self._portfolio_value()); self._prev_prices=None
            return self._get_obs(),-1.,done,False,{'turbulence':turb}
        prices=self.close_prices[self.current_step]
        if self._prev_prices is not None:
            pc=(prices-self._prev_prices)/(self._prev_prices+1e-8)
            for i in range(self.n_assets):
                if self.shares_held[i]<0 and pc[i]>0.02:
                    self.balance-=(-self.shares_held[i])*prices[i]*(1+self.commission); self.shares_held[i]=0
        self._prev_prices=prices.copy()
        sd=(action*self.max_shares).astype(np.float32)
        msv=self._portfolio_value()*self.margin_ratio
        for i in range(self.n_assets):
            if sd[i]<0: sd[i]=max(sd[i],-msv/(prices[i]+1e-8)-self.shares_held[i])
        buy=np.sum(np.maximum(sd,0)*prices)
        if buy>self.balance: sd=np.where(sd>0,sd*self.balance/(buy+1e-8),sd)
        cost=np.sum(np.abs(sd)*prices)*self.commission
        self.balance-=np.sum(sd*prices)+cost; self.shares_held+=sd
        self.trade_count+=int(np.any(sd!=0)); self.current_step+=1
        done=self.current_step>=len(self.close_prices)
        nv=self._portfolio_value(); self.portfolio_history.append(nv)
        nidx=min(self.current_step,len(self.close_prices)-1)
        pc2=(self.close_prices[nidx]-prices)/(prices+1e-8)
        raw=(nv-self.prev_portfolio_value)/(self.prev_portfolio_value+1e-8)
        raw+=np.sum(np.sign(self.shares_held)*pc2)*0.1
        raw-=np.sum((self.shares_held<0)*(pc2>0)*np.abs(self.shares_held)*prices)/(self.prev_portfolio_value+1e-8)*0.2
        r=float(np.clip(raw,-1.,1.)); self.prev_portfolio_value=nv
        return self._get_obs(),r,done,False,{'portfolio_value':nv,'turbulence':turb}
    def _portfolio_value(self):
        idx=min(self.current_step,len(self.close_prices)-1)
        return float(self.balance+np.sum(self.shares_held*self.close_prices[idx]))

WINDOW_SIZE=30
env_check=MOEXTradingEnvShort(close_prices=close_test,features=features_test,
    turbulence=turb_test_arr,turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE,margin_ratio=0.25)
obs,_=env_check.reset()
print(f'OK. Obs shape: {obs.shape} | Action: {env_check.action_space}')
del env_check

## Шаг 4: Четыре архитектуры

In [ ]:
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv
from sb3_contrib import RecurrentPPO
from xlstm import (xLSTMBlockStack,xLSTMBlockStackConfig,
    mLSTMBlockConfig,mLSTMLayerConfig,sLSTMBlockConfig,sLSTMLayerConfig,FeedForwardConfig)

def build_xlstm_stack(embedding_dim,context_length,num_blocks=4,use_slstm=False):
    if use_slstm:
        cfg=xLSTMBlockStackConfig(
            mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig(conv1d_kernel_size=4,qkv_proj_blocksize=4,num_heads=4)),
            slstm_block=sLSTMBlockConfig(
                slstm=sLSTMLayerConfig(backend='cuda',num_heads=4,conv1d_kernel_size=4,bias_init='powerlaw_blockdependent'),
                feedforward=FeedForwardConfig(proj_factor=1.3,act_fn='gelu')),
            context_length=context_length,num_blocks=num_blocks,embedding_dim=embedding_dim,slstm_at=[1])
    else:
        cfg=xLSTMBlockStackConfig(
            mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig(conv1d_kernel_size=4,qkv_proj_blocksize=4,num_heads=4)),
            context_length=context_length,num_blocks=num_blocks,embedding_dim=embedding_dim,slstm_at=[])
    return xLSTMBlockStack(cfg)

class LSTMEncoder(BaseFeaturesExtractor):
    """Baseline: обычный LSTM."""
    def __init__(self,observation_space,hidden_dim=128):
        super().__init__(observation_space,features_dim=hidden_dim)
        _,nf=observation_space.shape
        self.lstm=nn.LSTM(input_size=nf,hidden_size=hidden_dim,num_layers=1,batch_first=True)
    def forward(self,obs): out,_=self.lstm(obs); return out[:,-1,:]

class xLSTMEncoder(BaseFeaturesExtractor):
    """Вариант 1: базовый xLSTM (emb=128, blocks=4)."""
    def __init__(self,observation_space,embedding_dim=128,num_blocks=4,use_slstm=False):
        super().__init__(observation_space,features_dim=embedding_dim)
        ws,nf=observation_space.shape
        self.input_proj=nn.Sequential(nn.Linear(nf,embedding_dim),nn.GELU())
        self.xlstm=build_xlstm_stack(embedding_dim,ws,num_blocks,use_slstm)
    def forward(self,obs):
        x=self.input_proj(obs); x=self.xlstm(x); return x[:,-1,:]

class xLSTMEncoderWithAttention(BaseFeaturesExtractor):
    """
    Вариант 2: xLSTM + Self-Attention.
    После xLSTM добавлен MultiheadAttention по временным шагам.
    Модель сама решает какие дни в окне наблюдений важнее.
    Residual connection + LayerNorm стабилизируют обучение.
    """
    def __init__(self,observation_space,embedding_dim=128,num_blocks=4,num_heads=4,use_slstm=False):
        super().__init__(observation_space,features_dim=embedding_dim)
        ws,nf=observation_space.shape
        self.input_proj=nn.Sequential(nn.Linear(nf,embedding_dim),nn.GELU())
        self.xlstm=build_xlstm_stack(embedding_dim,ws,num_blocks,use_slstm)
        self.attn=nn.MultiheadAttention(embed_dim=embedding_dim,num_heads=num_heads,
                                        batch_first=True,dropout=0.1)
        self.norm=nn.LayerNorm(embedding_dim)
    def forward(self,obs):
        x=self.input_proj(obs)           # (batch, T, emb)
        x=self.xlstm(x)                 # (batch, T, emb)
        attn_out,_=self.attn(x,x,x)    # self-attention
        x=self.norm(x+attn_out)         # residual
        return x[:,-1,:]                # последний шаг

class xLSTMEncoderLarge(BaseFeaturesExtractor):
    """
    Вариант 3: xLSTM Large (emb=256, blocks=6).
    Больше ёмкость — сложнее паттерны.
    Требует больше GPU памяти — на Colab T4 может не хватить.
    """
    def __init__(self,observation_space,embedding_dim=256,num_blocks=6,use_slstm=False):
        super().__init__(observation_space,features_dim=embedding_dim)
        ws,nf=observation_space.shape
        self.input_proj=nn.Sequential(nn.Linear(nf,embedding_dim),nn.GELU())
        self.xlstm=build_xlstm_stack(embedding_dim,ws,num_blocks,use_slstm)
    def forward(self,obs):
        x=self.input_proj(obs); x=self.xlstm(x); return x[:,-1,:]

def check_slstm_available():
    if not torch.cuda.is_available(): return False
    return torch.cuda.get_device_capability()[0]>=8

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_SLSTM=check_slstm_available()
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    mj,mn=torch.cuda.get_device_capability()
    print(f'Compute Capability: {mj}.{mn} ({"sLSTM OK" if USE_SLSTM else "mLSTM only"})')

obs_space=MOEXTradingEnvShort(close_prices=close_test,features=features_test,
    turbulence=turb_test_arr,turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE).observation_space
dummy=torch.zeros(2,*obs_space.shape).to(DEVICE)
print('\nПроверка архитектур:')
for name,cls,kw in [
    ('LSTM baseline',  LSTMEncoder,             dict(hidden_dim=128)),
    ('xLSTM Base',     xLSTMEncoder,             dict(embedding_dim=128,num_blocks=4,use_slstm=USE_SLSTM)),
    ('xLSTM+Attention',xLSTMEncoderWithAttention,dict(embedding_dim=128,num_blocks=4,num_heads=4,use_slstm=USE_SLSTM)),
    ('xLSTM Large',    xLSTMEncoderLarge,        dict(embedding_dim=256,num_blocks=6,use_slstm=USE_SLSTM)),
]:
    enc=cls(obs_space,**kw).to(DEVICE); out=enc(dummy)
    print(f'  {name:<22}: output={out.shape}, params={sum(p.numel() for p in enc.parameters()):,}')
    del enc,out
del dummy

## Шаг 5: Curriculum Learning — обучение всех моделей

In [ ]:
BATCH_SIZE=32; LEARNING_RATE=1e-4; WINDOW_SIZE=30

def make_env(close,features,turb,window=30,margin=0.25):
    return DummyVecEnv([lambda: MOEXTradingEnvShort(
        close_prices=close,features=features,turbulence=turb,
        turbulence_threshold=TURBULENCE_THRESHOLD,
        window_size=window,margin_ratio=margin)])

def train_curriculum(model,name,steps1=300_000,steps2=700_000):
    """
    Этап 1: спокойный рынок 2013-2019 (model уже инициализирован на этой среде)
    Этап 2: полный период 2009-2021 с кризисами
    reset_num_timesteps=False — сохраняем накопленный опыт
    """
    print(f'\n[{name}] Этап 1: 2013-2019 ({steps1:,} шагов)...')
    model.learn(total_timesteps=steps1,progress_bar=True)
    model.set_env(make_env(close_train,features_train,turb_train_arr))
    print(f'[{name}] Этап 2: 2009-2021 ({steps2:,} шагов)...')
    model.learn(total_timesteps=steps2,progress_bar=True,reset_num_timesteps=False)
    model.save(f'model_{name}'); print(f'[{name}] Сохранено: model_{name}')
    return model

base_ppo=dict(
    policy='MlpLstmPolicy',
    env=make_env(close_all[calm_mask],features_all[calm_mask],turb_aligned[calm_mask].values),
    learning_rate=LEARNING_RATE,n_steps=2048,batch_size=BATCH_SIZE,n_epochs=10,
    gamma=0.99,gae_lambda=0.95,clip_range=0.1,ent_coef=0.02,vf_coef=0.5,
    max_grad_norm=0.5,verbose=1,seed=42,tensorboard_log='./tb_logs/',device=DEVICE)

# 1. LSTM baseline
print('='*55)
model_lstm=RecurrentPPO(**base_ppo,policy_kwargs=dict(
    features_extractor_class=LSTMEncoder,
    features_extractor_kwargs=dict(hidden_dim=128),
    lstm_hidden_size=128,n_lstm_layers=1,enable_critic_lstm=True,net_arch=[]))
model_lstm=train_curriculum(model_lstm,'lstm')

# 2. xLSTM Base
print('='*55)
model_base=RecurrentPPO(**base_ppo,policy_kwargs=dict(
    features_extractor_class=xLSTMEncoder,
    features_extractor_kwargs=dict(embedding_dim=128,num_blocks=4,use_slstm=USE_SLSTM),
    lstm_hidden_size=128,n_lstm_layers=1,enable_critic_lstm=True,net_arch=[]))
model_base=train_curriculum(model_base,'xlstm_base')

# 3. xLSTM + Attention
print('='*55)
model_attn=RecurrentPPO(**base_ppo,policy_kwargs=dict(
    features_extractor_class=xLSTMEncoderWithAttention,
    features_extractor_kwargs=dict(embedding_dim=128,num_blocks=4,num_heads=4,use_slstm=USE_SLSTM),
    lstm_hidden_size=128,n_lstm_layers=1,enable_critic_lstm=True,net_arch=[]))
model_attn=train_curriculum(model_attn,'xlstm_attn')

# 4. xLSTM Large (закомментируй на Colab T4 если не хватает памяти)
print('='*55)
model_large=RecurrentPPO(**base_ppo,policy_kwargs=dict(
    features_extractor_class=xLSTMEncoderLarge,
    features_extractor_kwargs=dict(embedding_dim=256,num_blocks=6,use_slstm=USE_SLSTM),
    lstm_hidden_size=256,  # совпадает с embedding_dim
    n_lstm_layers=1,enable_critic_lstm=True,net_arch=[]))
model_large=train_curriculum(model_large,'xlstm_large')

print('\nВсе модели обучены!')

## Шаг 6: Тестирование + итоговая таблица

In [ ]:
def run_episode(model,env):
    obs,_=env.reset(); obs=obs[np.newaxis,...]
    states=None; starts=np.array([True]); done=False
    while not done:
        action,states=model.predict(obs,state=states,episode_start=starts,deterministic=True)
        obs_raw,_,terminated,truncated,_=env.step(action[0])
        obs=obs_raw[np.newaxis,...]; starts=np.array([terminated or truncated])
        done=terminated or truncated
    return env.portfolio_history

def compute_metrics(hist,initial_balance,n_trades,rf=0.16):
    v=np.array(hist,dtype=float)
    CR=(v[-1]-initial_balance)/initial_balance*100
    MER=(v.max()-initial_balance)/initial_balance*100
    rm=np.maximum.accumulate(v)
    MPB=abs(((v-rm)/(rm+1e-8)).min())*100
    APPT=(v[-1]-initial_balance)/max(n_trades,1)/1000
    dr=np.diff(v)/(v[:-1]+1e-8); ex=dr-rf/252
    SR=(ex.mean()/(ex.std()+1e-8))*np.sqrt(252)
    return {'CR':CR,'MER':MER,'MPB':MPB,'APPT':APPT,'SR':SR}

def make_test_env():
    return MOEXTradingEnvShort(close_prices=close_test,features=features_test,
        turbulence=turb_test_arr,turbulence_threshold=TURBULENCE_THRESHOLD,
        window_size=WINDOW_SIZE,margin_ratio=0.25)

all_models=[('LSTM baseline',model_lstm),('xLSTM Base',model_base),
            ('xLSTM+Attention',model_attn),('xLSTM Large',model_large)]
results={}; portfolios={}

for name,model in all_models:
    print(f'Тестируем {name}...')
    env=make_test_env(); hist=run_episode(model,env)
    m=compute_metrics(hist,env.initial_balance,env.trade_count)
    results[name]=m; portfolios[name]=np.array(hist)
    print(f'  CR={m["CR"]:+.2f}%  SR={m["SR"]:.3f}  MPB={m["MPB"]:.2f}%')

print('\n'+'='*68)
print(f'{"Метрика":<10}',end='')
for n in results: print(f'{n:>14}',end='')
print(); print('-'*68)
for k in ['CR','MER','MPB','APPT','SR']:
    print(f'{k:<10}',end='')
    for n in results: print(f'{results[n][k]:>14.3f}',end='')
    print()
print('='*68)
best=max(results,key=lambda k:results[k]['SR'])
print(f'\nЛучшая по SR: {best}  SR={results[best]["SR"]:.3f}  CR={results[best]["CR"]:+.2f}%')

## Шаг 7: Визуализация

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

IB=1_000_000.
COLORS={'LSTM baseline':'#888888','xLSTM Base':'#3266ad',
        'xLSTM+Attention':'#1D9E75','xLSTM Large':'#BA7517'}
STYLES={'LSTM baseline':'--','xLSTM Base':'-','xLSTM+Attention':'-','xLSTM Large':'-'}
WIDTHS={'LSTM baseline':1.2,'xLSTM Base':1.5,'xLSTM+Attention':2.2,'xLSTM Large':1.5}

fig,axes=plt.subplots(2,1,figsize=(14,9))
fig.suptitle('Сравнение архитектур: LSTM vs xLSTM Base vs xLSTM+Attn vs xLSTM Large\n'
             'MOEX (тест 2022-2024), Short стратегия, Curriculum Learning',
             fontsize=13,fontweight='bold')

ax=axes[0]
for name,portfolio in portfolios.items():
    ret=(portfolio/IB-1)*100; days=np.arange(len(ret)); m=results[name]
    ax.plot(days,ret,label=f"{name}  CR={m['CR']:+.1f}%  SR={m['SR']:.2f}",
            color=COLORS[name],linestyle=STYLES[name],linewidth=WIDTHS[name])
ax.axhline(y=0,color='black',linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylabel('Накопленная доходность (%)'); ax.set_title('Накопленная доходность')
ax.legend(loc='upper left',fontsize=9); ax.grid(True,alpha=0.3)

ax=axes[1]
for name,portfolio in portfolios.items():
    rm=np.maximum.accumulate(portfolio); dd=(portfolio-rm)/(rm+1e-8)*100
    m=results[name]
    ax.plot(np.arange(len(dd)),dd,label=f"{name}  MPB={m['MPB']:.1f}%",
            color=COLORS[name],linestyle=STYLES[name],linewidth=WIDTHS[name])
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylabel('Drawdown (%)'); ax.set_xlabel('Торговый день'); ax.set_title('Просадка')
ax.legend(loc='lower left',fontsize=9); ax.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('architecture_comparison.png',dpi=150,bbox_inches='tight')
plt.show(); print('Сохранено: architecture_comparison.png')

# Детальный график лучшей модели
best=max(results,key=lambda k:results[k]['SR'])
port=portfolios[best]; days=np.arange(len(port))
ret_pct=(port/IB-1)*100; dr=np.diff(port)/port[:-1]*100

fig,axes=plt.subplots(3,1,figsize=(14,10),sharex=False)
fig.suptitle(f'Лучшая модель: {best} — MOEX (тест 2022-2024)',
             fontsize=14,fontweight='bold',y=0.98)

ax=axes[0]
ax.plot(days,port/1e6,color=COLORS[best],linewidth=1.5,label='Портфель')
ax.axhline(y=IB/1e6,color='gray',linestyle='--',linewidth=1,label='Начальный капитал')
ax.set_ylabel('Стоимость (млн ₽)'); ax.set_title('Динамика портфеля')
ax.legend(); ax.grid(True,alpha=0.3)

ax=axes[1]
ax.plot(days,ret_pct,color=COLORS[best],linewidth=1.5)
ax.axhline(y=0,color='black',linewidth=0.8)
ax.fill_between(days,ret_pct,0,where=(ret_pct>=0),alpha=0.2,color='seagreen')
ax.fill_between(days,ret_pct,0,where=(ret_pct<0),alpha=0.2,color='tomato')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylabel('Доходность (%)'); ax.set_title('Накопленная доходность'); ax.grid(True,alpha=0.3)

ax=axes[2]
cb=['seagreen' if r>=0 else 'tomato' for r in dr]
ax.bar(np.arange(len(dr)),dr,color=cb,alpha=0.7,width=1.0)
ax.axhline(y=0,color='black',linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel('Торговый день'); ax.set_ylabel('Дневная доходность (%)')
ax.set_title('Дневные доходности'); ax.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig(f'best_{best.replace(" ","_")}.png',dpi=150,bbox_inches='tight')
plt.show(); print(f'Сохранено: best_{best.replace(" ","_")}.png')